# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [2]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [5]:
# TODO
# GridSearchCV를 수행하세요.
model = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4,8,12]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=grid_params,
    cv=5,
    scoring='r2'
)

# 탐색 시간 측정
start = time.time()
grid_search.fit(X_train,y_train)
grid_time = time.time() - start

print(f'GridSearchCV 탐색 시간: {grid_time:.2f}초')
print('평가한 조합 수', len(grid_search.cv_results_['params']))


print('best_params_:',grid_search.best_params_)
print(f'best_score_ : {grid_search.best_score_}')


grid_results = pd.DataFrame(grid_search.cv_results_)
best_idx = grid_search.best_index_

best_mean = grid_results.loc[best_idx, 'mean_test_score']
best_std = grid_results.loc[best_idx, 'std_test_score']

print(f'최적 조합 mean_test_score: {best_mean:.4f}')
print(f'최적 조합 std_test_score: {best_std:.4f}')


display_cols = ["param_n_estimators", "param_max_depth",
                "mean_test_score", "std_test_score", "rank_test_score"]
display(grid_results[display_cols].sort_values("rank_test_score"))

GridSearchCV 탐색 시간: 6.44초
평가한 조합 수 9
best_params_: {'max_depth': 12, 'n_estimators': 200}
best_score_ : 0.8206179847869421
최적 조합 mean_test_score: 0.8206
최적 조합 std_test_score: 0.0644


,param_n_estimators,param_max_depth,mean_test_score,std_test_score,rank_test_score
7,200,12,0.820618,0.064425,1
6,100,12,0.820348,0.065524,2
8,300,12,0.820182,0.065787,3
5,300,8,0.818850,0.063274,4
4,200,8,0.817849,0.064449,5
3,100,8,0.817761,0.065969,6
2,300,4,0.780580,0.050523,7
1,200,4,0.779586,0.049544,8
0,100,4,0.779369,0.051308,9


- 최고 CV 평균
    - 0.82, max_depth(12)와 n_estimators(200)의 조합으로 5개의 검증 R²을 평균한 값

- CV 표준편차
    - 0.06, 최적 조합 Fold별로 검증 점수가 평균 주변에서 얼마나 달랐는지를 나타냄

### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?
**답변:**  
-  장점: 지정한 조합을 빠짐없이 평가, 하이퍼 파라미터 튜닝 최적화를 용이하게 찾음
-  단점: 후보 수가 많아지면 탐색비용이 매우 느려짐

## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [15]:
# TODO
# RandomizedSearchCV를 수행하세요.
random_params = {
    'n_estimators': randint(100, 301),
    'max_depth': randint(4, 13)
}

random_search = RandomizedSearchCV(
    estimator = model,
    param_distributions = random_params,
    n_iter = 9,
    cv=5,
    scoring='r2',
    random_state=42  
)

start = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start

print(f'RandomizedSearchCV 탐색 시간: {finish:.2f}초')

print('최적 파라미터:', random_search.best_params_)
print(f'cv 평균 점수: {random_search.best_score_:.4f}')

random_results = pd.DataFrame(random_search.cv_results_)
best_idx = random_search.best_index_

print(f'최적 조합 mean_test_score: {random_results.loc[best_idx, 'mean_test_score']:.4f}')
print(f'최적 조합 std_test_score: {random_results.loc[best_idx, 'std_test_score']:.4f}')

display_cols = ["param_n_estimators", "param_max_depth",
                "mean_test_score", "std_test_score", "rank_test_score"]
display(random_results[display_cols].sort_values("rank_test_score"))


RandomizedSearchCV 탐색 시간: 6.84초
최적 파라미터: {'max_depth': 11, 'n_estimators': 230}
cv 평균 점수: 0.8210
최적 조합 mean_test_score: 0.8210
최적 조합 std_test_score: 0.0651


,param_n_estimators,param_max_depth,mean_test_score,std_test_score,rank_test_score
6,230,11,0.821030,0.065086,1
1,288,11,0.820919,0.066358,2
4,216,11,0.819858,0.065923,3
0,279,10,0.818594,0.069279,4
2,202,8,0.818300,0.063981,5
7,152,9,0.817948,0.067975,6
5,203,7,0.816651,0.062739,7
3,174,6,0.809847,0.061638,8
8,187,5,0.801043,0.055673,9


- Grid 서치의 CV 평균보다 약 0.0004높다.
- 반면에 표준편차는 0.006 정도로 커졌다.
- Grid 서치와 지금 모델을 비교하여 보면 평균 R²은 높지만 표준편차(변동)가 조금 더 크다

### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?
**답변:**
* 내가 어떤 값을 넣어야 할 지 모르고 모든 조합을 평가할 시간이 없을 때 주로 유용하다.  
<br> GridSearch에서는 모든 조합을 탐색하기 때문에 탐색시간이 증가하지만 Random Search는 무작위로 조합을 선택해서 n_iter로 지정한 횟수만큼만 탐색할 수 있기 때문이다.

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [13]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.

# Optuna objective 함수 정의
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 300)
    max_depth = trial.suggest_int('max_depth',4,12)

    # 모델 생성
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    trial.set_user_attr('std_test_score',scores.std())

    return scores.mean()


# TPESampler 설정
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)

# study
study = optuna.create_study(
    direction='maximize',
    sampler= sampler
)

start = time.time()
study.optimize(objective, n_trials=9)
optuna_time = time.time() - start

print(f'Optuna 탐색 시간: {optuna_time:.2f}초')

print("best_params:", study.best_params)
print(f"best_value : {study.best_value:.4f}")
print(f"best std   : {study.best_trial.user_attrs['std_test_score']:.4f}")

[I 2026-10-07 18:05:58,369] A new study created in memory with name: no-name-91f6afc1-5c47-471e-a77a-56faa883f959
[I 2026-10-07 18:05:59,041] Trial 0 finished with value: 0.8212542415716282 and parameters: {'n_estimators': 175, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 18:05:59,872] Trial 1 finished with value: 0.8184595590761907 and parameters: {'n_estimators': 247, 'max_depth': 9}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 18:06:00,369] Trial 2 finished with value: 0.7990289774524457 and parameters: {'n_estimators': 131, 'max_depth': 5}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 18:06:00,795] Trial 3 finished with value: 0.8201477777785264 and parameters: {'n_estimators': 109, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 18:06:01,540] Trial 4 finished with value: 0.8205950100254196 and parameters: {'n_estimators': 204, 'max_depth': 12}. Best is trial 0 with value: 0.821254

Optuna 탐색 시간: 6.04초
best_params: {'n_estimators': 175, 'max_depth': 12}
best_value : 0.8213
best std   : 0.0643


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?
- Grid는 정해 둔 모든 후보 조합을 평가하고, Random은 정해진 횟수만큼 후보를 무작위로 선택
- Optuna는 초기 3개 trial 이후 결과를 참고하여 상대적으로 유리한 영역을 더 깊게 탐색하는 방식을 사용

<br> Optuna가 Grid / Random과 다른 핵심은 Optuna는 이전 Trial의 성능 결과를 참고하여 성능이 좋을 것으로 예상되는 하이퍼파라미터를 우선적으로 탐색한다는 점이다.

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [14]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.
# 1. best_params로 모델 생성
optuna_best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

# 2. 5-Fold 교차검증 재수행
optuna_scores = cross_val_score(
    optuna_best_model,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

# 3. 평균과 표준편차 계산
optuna_mean = optuna_scores.mean()
optuna_std = optuna_scores.std()

print("fold별 점수:", optuna_scores.round(4))
print(f"CV 평균     : {optuna_mean:.4f}")
print(f"CV 표준편차 : {optuna_std:.4f}")

# 참고: 탐색 중 기록된 값과 비교
print(f"study.best_value와 차이: {abs(optuna_mean - study.best_value):.6f}")

fold별 점수: [0.8694 0.6967 0.828  0.8428 0.8694]
CV 평균     : 0.8213
CV 표준편차 : 0.0643
study.best_value와 차이: 0.000000


- 앞서 진행한 세 가지 모델 결고 비교
- Grid의 표준편차는 약 0.064425, Random은 0.06508였고 Optuna는 0.0642이다.
- Optuna를 통해 결과를 구했을 때 평균이 가장 높고, 표준편차가 가장 적었다.


---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [16]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.

from sklearn.model_selection import KFold, cross_validate

# =========================================================
# 1. 세 탐색 결과를 하나의 DataFrame으로 정리
# =========================================================
def to_builtin(params):
    """numpy 정수를 파이썬 int로 바꿔 JSON 저장이 가능하게 함"""
    return {k: int(v) for k, v in params.items()}

search_summary = pd.DataFrame([
    {
        "search_method": "GridSearchCV",
        "best_params": to_builtin(grid_search.best_params_),
        "cv_mean": grid_search.best_score_,
        "cv_std": grid_search.cv_results_["std_test_score"][grid_search.best_index_],
        "search_seconds": grid_time,
    },
    {
        "search_method": "RandomizedSearchCV",
        "best_params": to_builtin(random_search.best_params_),
        "cv_mean": random_search.best_score_,
        "cv_std": random_search.cv_results_["std_test_score"][random_search.best_index_],
        "search_seconds": random_time,
    },
    {
        "search_method": "Optuna(TPE)",
        "best_params": to_builtin(study.best_params),
        "cv_mean": optuna_mean,
        "cv_std": optuna_std,
        "search_seconds": optuna_time,
    },
])

search_summary = (search_summary
                  .sort_values("cv_mean", ascending=False)
                  .reset_index(drop=True))
print("=== 세 탐색 전략 비교 ===")
print(search_summary.to_string())

# =========================================================
# 2~3. 튜닝 후보 선택
#   - CV 평균 1순위
#   - 1위와의 평균 차이가 MEAN_TOL 이내인 후보끼리는
#     CV 표준편차가 작은 쪽 → 탐색 시간이 짧은 쪽 순으로 선택
# =========================================================
MEAN_TOL = 0.002   # R² 0.002 이내면 "거의 같은 성능"으로 간주

top_mean = search_summary["cv_mean"].max()
search_summary["gap_from_top"] = top_mean - search_summary["cv_mean"]

close = search_summary[search_summary["gap_from_top"] <= MEAN_TOL]
selected = close.sort_values(["cv_std", "search_seconds"]).iloc[0]

selected_method = selected["search_method"]
selected_params = selected["best_params"]

print("\n=== 튜닝 후보 선택 ===")
if len(close) == 1:
    print(f"{selected_method}의 CV 평균({selected['cv_mean']:.4f})이 가장 높고, "
          f"다른 후보와의 차이가 {MEAN_TOL} 보다 커서 평균 기준으로 선택했습니다.")
else:
    print(f"CV 평균 차이가 {MEAN_TOL} 이내인 후보가 {len(close)}개 → "
          f"표준편차와 탐색 시간을 함께 비교했습니다.")
    print(close[["search_method", "cv_mean", "cv_std", "search_seconds"]].to_string())
    print(f"→ 표준편차({selected['cv_std']:.4f})가 가장 작은 "
          f"{selected_method}를 선택했습니다.")
print("선택 파라미터:", selected_params)

# =========================================================
# 4. 기준 모델 vs 튜닝 후보 (동일 Fold)
# =========================================================
kf = KFold(n_splits=5, shuffle=False)   # cv=5와 동일한 분할

baseline_params = {"n_estimators": 100, "max_depth": None,
                   "random_state": 42, "n_jobs": -1}
tuned_params = {**selected_params, "random_state": 42, "n_jobs": -1}

cv_baseline = cross_validate(
    RandomForestRegressor(**baseline_params), X_train, y_train,
    cv=kf, scoring="r2", return_train_score=True
)
cv_tuned = cross_validate(
    RandomForestRegressor(**tuned_params), X_train, y_train,
    cv=kf, scoring="r2", return_train_score=True
)

# --- Fold별 비교 ---
fold_comparison = pd.DataFrame({
    "fold": range(1, 6),
    "baseline_train_r2": cv_baseline["train_score"],
    "baseline_val_r2": cv_baseline["test_score"],
    "tuned_train_r2": cv_tuned["train_score"],
    "tuned_val_r2": cv_tuned["test_score"],
})
fold_comparison["val_diff(tuned-baseline)"] = (
    fold_comparison["tuned_val_r2"] - fold_comparison["baseline_val_r2"]
)
print("\n=== fold_comparison ===")
print(fold_comparison.round(4).to_string(index=False))

# --- 요약 비교 ---
def summarize(name, res):
    train_mean = res["train_score"].mean()
    val_mean = res["test_score"].mean()
    return {
        "model": name,
        "train_r2_mean": train_mean,
        "val_r2_mean": val_mean,
        "val_r2_std": res["test_score"].std(),
        "train_val_gap": train_mean - val_mean,
    }

tuning_comparison = pd.DataFrame([
    summarize("baseline", cv_baseline),
    summarize("tuned", cv_tuned),
]).set_index("model")
print("\n=== tuning_comparison ===")
print(tuning_comparison.round(4).to_string())

# --- 튜닝 효과 (R² 차이) ---
r2_diff = (tuning_comparison.loc["tuned", "val_r2_mean"]
           - tuning_comparison.loc["baseline", "val_r2_mean"])
fold_wins = int((fold_comparison["val_diff(tuned-baseline)"] > 0).sum())

print(f"\n튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R² = {r2_diff:+.4f}")
print(f"튜닝 후보가 더 높은 fold 수: {fold_wins}/5")

# 동일 분할 확인: 탐색 때의 CV 평균과 같아야 함
print(f"(확인) 탐색 시 CV 평균과의 차이: "
      f"{abs(tuning_comparison.loc['tuned', 'val_r2_mean'] - selected['cv_mean']):.6f}")

# --- 최종 모델 확정 (Test 보기 전) ---
if r2_diff > 0:
    final_name = "tuned"
    final_params = tuned_params
    final_method = selected_method
else:
    final_name = "baseline"
    final_params = baseline_params
    final_method = "baseline (no tuning)"

final_cv_mean = tuning_comparison.loc[final_name, "val_r2_mean"]
final_cv_std = tuning_comparison.loc[final_name, "val_r2_std"]

b = tuning_comparison.loc["baseline"]
t = tuning_comparison.loc["tuned"]
print("\n=== 최종 모델 확정 ===")
print(f"검증 평균: baseline {b.val_r2_mean:.4f} vs tuned {t.val_r2_mean:.4f} "
      f"(차이 {r2_diff:+.4f}, {fold_wins}/5 fold에서 tuned 우세)")
print(f"검증 표준편차: baseline {b.val_r2_std:.4f} vs tuned {t.val_r2_std:.4f}")
print(f"Train-검증 차이: baseline {b.train_val_gap:.4f} vs tuned {t.train_val_gap:.4f}")
print(f"→ 최종 선택: {final_name} ({final_method})")

# 최종 모델을 Train 전체로 재학습
final_model = RandomForestRegressor(**final_params)
final_model.fit(X_train, y_train)

# =========================================================
# 5. hold-out Test 평가 (딱 한 번)
# =========================================================
test_r2 = final_model.score(X_test, y_test)
print(f"\nhold-out Test R²: {test_r2:.4f}")

# =========================================================
# 6. CV 평균과 Test R² 차이
# =========================================================
cv_test_gap = final_cv_mean - test_r2
print(f"CV 평균 - Test R² = {cv_test_gap:+.4f}  (CV 표준편차 {final_cv_std:.4f})")

if abs(cv_test_gap) <= final_cv_std:
    print("→ 차이가 CV 표준편차 이내: fold 간 변동 범위 안의 차이로 볼 수 있습니다.")
elif cv_test_gap > 0:
    print("→ Test가 CV보다 표준편차 이상 낮음: 과적합 또는 분할 차이를 의심해 볼 수 있습니다.")
else:
    print("→ Test가 CV보다 표준편차 이상 높음: Test 분할이 상대적으로 쉬웠을 가능성이 있습니다.")

# =========================================================
# 7~8. 모델과 메타데이터 저장
# =========================================================
os.makedirs("model_artifacts", exist_ok=True)

model_path = "model_artifacts/best_model.pkl"
joblib.dump(final_model, model_path)

meta = {
    "model_name": "RandomForestRegressor",
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "selected_search_method": final_method,
    "best_params": final_params,
    "cv_mean": float(final_cv_mean),
    "cv_std": float(final_cv_std),
    "test_r2": float(test_r2),
    "baseline_config": baseline_params,
    "tuned_candidate": {
        "search_method": selected_method,
        "params": tuned_params,
    },
    "search_summary": search_summary.to_dict(orient="records"),
    "tuning_comparison": tuning_comparison.reset_index().to_dict(orient="records"),
    "fold_comparison": fold_comparison.to_dict(orient="records"),
    "cv_r2_diff_tuned_minus_baseline": float(r2_diff),
    "cv_mean_minus_test_r2": float(cv_test_gap),
    "features": features,
}

with open("model_artifacts/model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

print("\n저장 완료:", model_path, "/ model_artifacts/model_meta.json")


=== 세 탐색 전략 비교 ===
        search_method                             best_params   cv_mean    cv_std  search_seconds
0         Optuna(TPE)  {'n_estimators': 175, 'max_depth': 12}  0.821254  0.064281        6.043230
1  RandomizedSearchCV  {'max_depth': 11, 'n_estimators': 230}  0.821030  0.065086        6.855922
2        GridSearchCV  {'max_depth': 12, 'n_estimators': 200}  0.820618  0.064425        6.436980

=== 튜닝 후보 선택 ===
CV 평균 차이가 0.002 이내인 후보가 3개 → 표준편차와 탐색 시간을 함께 비교했습니다.
        search_method   cv_mean    cv_std  search_seconds
0         Optuna(TPE)  0.821254  0.064281        6.043230
1  RandomizedSearchCV  0.821030  0.065086        6.855922
2        GridSearchCV  0.820618  0.064425        6.436980
→ 표준편차(0.0643)가 가장 작은 Optuna(TPE)를 선택했습니다.
선택 파라미터: {'n_estimators': 175, 'max_depth': 12}

=== fold_comparison ===
 fold  baseline_train_r2  baseline_val_r2  tuned_train_r2  tuned_val_r2  val_diff(tuned-baseline)
    1             0.9717           0.8627          0.9715        0.8694 

### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  
아니다. CV 평균이 조금 높더라도 탐색 시간이 크게 증가한다면 성능 향상이 탐색 비용을 감수할 만큼 충분한지 함께 고려해야 한다.
따라서 cv 평균뿐만 아니라 cv 표준편차, 탐색 시간, 실제 성능 향상 폭 등을 종합적으로 비교해야 한다.

### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  
- Test 데이터를 학습 과정이나 검증과정에 사용하면 모델이나 하이퍼파라미터를 수정하는 과정에서 Test 데이터에 맞춰 모델을 선택하게 되어, Test 성능이 실제 새로운 데이터에 대한 일반화 성능을 제대로 나타내지 못할 수 있기 때문이다.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:**
- train_r2_mean 은 0.9737 -> 0.9727로 0.0010 감소했고 val_r2_mean은 0.8190 -> 0.8213로 0.0023 증가했다. 따라서 train_val_gap은 0.1547 -> 0.1514로 0.0033 감소했다. 
- fold 별로 val_diff를 토대로 보면 1번(0.0067), 2번(0.0159) fold에서 튜닝한 val값이 높게 나왔고 3번, 4번,5번 fold에서는 오히려 baseline보다 낮았다. 따라서 모든 Fold에서 성능이 개선된 것은 아니다.
- 다만 CV 표준편차는 0.0704 → 0.0643으로 감소했고, Train-검증 점수 차이도 0.1547 → 0.1514로 감소했다. 따라서 평균 성능은 소폭 개선되었으며, Fold별 성능 변동과 과적합 정도도 약간 줄어들어 전반적으로 튜닝 후 모델이 개선되었다고 판단할 수 있다.  

### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.
- 튜닝 후 cv 평균 R²는 0.8213, 최종 hold-out test 결과 R²는 0.8911로 Test 성능이 cv 평균보다 높게 나타났다.
- 튜닝 후 Train R² 평균은 0.9727, Validation R² 평균은 0.8213으로 Train-검증 차이가 0.1514이므로 과적합 가능성이 완전히 없다고 볼 수는 없다. 다만 튜닝 전보다 Train-검증 차이가 0.1547 → 0.1514로 감소했고, CV 표준편차도 0.0704 → 0.0643으로 감소했다.
- 따라서 튜닝 후 모델의 cv 성능과 안정성은 소폭 개선되었지만 Test R²가 CV 평균보다 높다는 사실만으로 일반화 성능이 매우 좋다고 단정할 수는 없다. Test 데이터는 최종 평가에 한 번만 사용되었기 때문에, 이 Test 결과가 다른 새로운 데이터에서도 그대로 유지되는지는 확인할 수 없다는 한계가 있다.

---

# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?
<br> -> Grid -> 주어진 모든 후보 조합 평가
<br> -> Random -> n_iter 만큼 무작위 조합 평가

2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?
<br> -> Random Search가 평가할 조합의 개수, 탐색 예산을 의미

3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?
<br> -> study -> 전체 탐색
<br> -> trial -> 한 번의 시도

4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?
<br> -> cv 평균, cv 표준편차, 탐색 학습 비용

5. hold-out Test는 언제 사용해야 하나요?
<br> -> 모든 탐색과 모델 선택이 끝난 뒤 마지막에 한 번 사용한다.

6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?
<br> -> 나중에 같은 모델을 재사용하고
<br> -> 어떤 조건으로 만들어졌는 지 추적(Log)하여 재현성과 운영 관리를 높이기 위해 저장한다.